# Step-by-step PhAST problem setup

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CEMS-Lab/PhAST/blob/ea6a90f1ae22d8fdd69cc41d6e197d4229977968/docs/tutorial/problem_setup_walkthrough.ipynb)

This notebook builds a small single-edge-notched tension problem from first principles. It creates and inspects the geometry and mesh, defines the material and boundary-value problem with the fluent Python API, writes a runnable schema-v1 YAML configuration, performs a bounded two-step CPU solve, and inspects the resulting artifacts.

Learning objectives:

1. Create a Gmsh `.geo` file and convert it to a `.msh` mesh.
2. Verify physical groups before using them as PhAST regions.
3. Apply material parameters, initial-condition choices, supports, and prescribed loading.
4. Distinguish the fluent authoring representation from the runnable YAML contract.
5. Run a short quasi-static phase-field workflow check and inspect the generated output directory.
6. Locate final field images, histories, trajectory stores, and optional animations.

The default run is intentionally short and is not crack-growth validation. For a research calculation, start from the closest checked-in example, refine the mesh, select a documented loading schedule, and retain the full result and provenance bundle.


## 1. Install dependencies

Run this cell in Colab. In a local clone, install PhAST with `pip install -e .` from the repository root and skip the Colab-specific package installation if your environment is already configured.

The notebook uses Gmsh to create `mesh.msh`, MeshIO to inspect physical groups, Matplotlib/ImageIO for visual checks, and PhAST for solving and result inspection.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gmsh", "ffmpeg"], check=True)
    checkout = Path("/content/PhAST")
    if not checkout.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/CEMS-Lab/PhAST.git", str(checkout),
        ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-e", str(checkout),
        "meshio", "matplotlib", "zarr", "h5py", "pyyaml",
    ], check=True)
    os.chdir(checkout)
else:
    print("Local run: use an activated PhAST environment and start Jupyter from the repository checkout.")


## 2. Imports and working directory

All generated files go into `runs/notebook_sent/`. Keeping the generated geometry, mesh, configuration, and results in one directory makes the run easy to archive or remove.

In [ ]:
import json
import shutil
import textwrap

from PIL import Image as PILImage
import matplotlib.pyplot as plt
import meshio
import numpy as np
import yaml

here = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (here, *here.parents)
     if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "phast").is_dir()),
    here,
)
source_src = repo_root / "src"
if source_src.is_dir():
    sys.path.insert(0, str(source_src))
    os.environ["PYTHONPATH"] = str(source_src) + os.pathsep + os.environ.get("PYTHONPATH", "")

import phast

run_dir = (repo_root / "runs" / "notebook_sent").resolve()
run_dir.mkdir(parents=True, exist_ok=True)

geo_path = run_dir / "mesh.geo"
mesh_path = run_dir / "mesh.msh"
spec_path = run_dir / "authored_problem_spec.yaml"
config_path = run_dir / "config.yaml"
output_dir = run_dir / "results"

print("PhAST:", getattr(phast, "__version__", "source checkout"))
print("Repository root:", repo_root)
print("Working directory:", run_dir)


## 3. Define a small geometry

The geometry is a rectangular plate with a thin single-edge notch cut from the left boundary to mid-plate. The notch boundary is a named physical curve that supplies nodes for the initial damage field. This makes the geometry robust in Gmsh while still representing a pre-existing crack seed for the phase-field solve.

The physical names are the contract between the mesh and PhAST:

| Name | Meaning | Used for |
|---|---|---|
| `body` | 2D plate surface | material assignment |
| `bottom` | bottom edge | fixed support |
| `top` | top edge | prescribed vertical displacement |
| `notch` | embedded crack line | initial damage seed |
| `left`, `right` | side edges | optional inspection/output regions |

In [ ]:
L = 1.0          # plate width [mm]
H = 1.0          # plate height [mm]
a0 = 0.50        # initial notch/crack length [mm]
y_crack = 0.50   # crack vertical location [mm]
notch_gap = 0.012 # small geometric opening used for robust meshing [mm]
h_bulk = 0.08    # coarse element size [mm]
h_crack = 0.02   # local element size near the crack [mm]

geo_text = f"""
SetFactory("OpenCASCADE");

L = {L};
H = {H};
a0 = {a0};
yc = {y_crack};
g = {notch_gap};
h_bulk = {h_bulk};
h_crack = {h_crack};

Point(1) = {{0, 0, 0, h_bulk}};
Point(2) = {{L, 0, 0, h_bulk}};
Point(3) = {{L, H, 0, h_bulk}};
Point(4) = {{0, H, 0, h_bulk}};
Point(5) = {{0, yc + 0.5*g, 0, h_crack}};
Point(6) = {{a0, yc + 0.5*g, 0, h_crack}};
Point(7) = {{a0, yc - 0.5*g, 0, h_crack}};
Point(8) = {{0, yc - 0.5*g, 0, h_crack}};

Line(1) = {{1, 2}};
Line(2) = {{2, 3}};
Line(3) = {{3, 4}};
Line(4) = {{4, 5}};
Line(5) = {{5, 6}};
Line(6) = {{6, 7}};
Line(7) = {{7, 8}};
Line(8) = {{8, 1}};

Curve Loop(10) = {{1, 2, 3, 4, 5, 6, 7, 8}};
Plane Surface(20) = {{10}};

Field[1] = Distance;
Field[1].CurvesList = {{5, 6, 7}};
Field[2] = Threshold;
Field[2].InField = 1;
Field[2].SizeMin = h_crack;
Field[2].SizeMax = h_bulk;
Field[2].DistMin = 0.02;
Field[2].DistMax = 0.18;
Background Field = 2;

Physical Surface("body") = {{20}};
Physical Curve("bottom") = {{1}};
Physical Curve("right") = {{2}};
Physical Curve("top") = {{3}};
Physical Curve("left") = {{4, 8}};
Physical Curve("notch") = {{5, 6, 7}};
""".strip()

geo_path.write_text(geo_text + "\n", encoding="utf-8")
print(geo_path)
print(geo_text[:600] + "\n...")

## 4. Visualize the geometry before meshing

This schematic is not the finite-element mesh. It is a cheap sanity check: dimensions, crack location, and loaded/support boundaries should be correct before generating elements.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, L, L, 0, 0], [0, 0, H, H, 0], color="0.15", lw=2)
ax.fill([0, a0, a0, 0], [y_crack - notch_gap/2, y_crack - notch_gap/2, y_crack + notch_gap/2, y_crack + notch_gap/2], color="crimson", alpha=0.22)
ax.plot([0, a0], [y_crack, y_crack], color="crimson", lw=3, label="initial notch / damage seed")
ax.plot([0, L], [0, 0], color="royalblue", lw=4, label="fixed bottom")
ax.plot([0, L], [H, H], color="darkorange", lw=4, label="prescribed top displacement")
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("Geometry and named boundaries")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1)
fig.tight_layout()
fig.savefig(run_dir / "geometry_preview.png", dpi=180)
plt.show()

## 5. Generate the mesh with Gmsh

The command below converts `mesh.geo` into `mesh.msh`. If this fails in a local environment, install Gmsh and make sure the `gmsh` executable is on `PATH`.

In [ ]:
gmsh = shutil.which("gmsh")
if gmsh is None:
    raise RuntimeError("gmsh executable was not found on PATH")

cmd = [gmsh, "-2", str(geo_path), "-format", "msh2", "-o", str(mesh_path)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(mesh_path, mesh_path.stat().st_size, "bytes")

## 6. Inspect named regions from the mesh

This is the most important mesh validation step. The solver can only apply materials, initial conditions, supports, loads, and histories to regions that are present in the mesh. Missing or misspelled physical names should be fixed in the `.geo` file before running.

In [ ]:
mesh_summary = phast.inspect_mesh(mesh_path)
print("points:", mesh_summary["n_points"])
print("cell blocks:", mesh_summary["cells"])
print(json.dumps(mesh_summary["named_groups"], indent=2))

required_groups = {"body", "bottom", "top", "notch"}
missing = required_groups - set(mesh_summary["named_groups"])
if missing:
    raise RuntimeError(f"Missing required physical groups: {sorted(missing)}")

## 7. Visualize the mesh and physical groups

A visual check catches common mistakes: swapped top/bottom groups, an unembedded crack line, very coarse crack-tip resolution, or an accidental empty surface group.

In [ ]:
mesh = meshio.read(mesh_path)
points = mesh.points[:, :2]

triangles = []
lines = []
line_tags = []
for block_index, block in enumerate(mesh.cells):
    if block.type == "triangle":
        triangles.append(block.data)
    if block.type == "line":
        lines.append(block.data)
        tags = mesh.cell_data.get("gmsh:physical", [])[block_index]
        line_tags.append(tags)

triangles = np.vstack(triangles) if triangles else np.empty((0, 3), dtype=int)
lines = np.vstack(lines) if lines else np.empty((0, 2), dtype=int)
line_tags = np.concatenate(line_tags) if line_tags else np.empty((0,), dtype=int)

id_to_name = {int(data[0]): name for name, data in mesh.field_data.items()}
colors = {"bottom": "royalblue", "top": "darkorange", "left": "0.35", "right": "0.35", "notch": "crimson"}

fig, ax = plt.subplots(figsize=(7, 6))
if len(triangles):
    ax.triplot(points[:, 0], points[:, 1], triangles, color="0.82", linewidth=0.5)
for edge, tag in zip(lines, line_tags):
    name = id_to_name.get(int(tag), str(tag))
    xy = points[edge]
    ax.plot(xy[:, 0], xy[:, 1], color=colors.get(name, "0.4"), lw=2.0 if name == "notch" else 1.4)
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("Mesh with named physical curves")
fig.tight_layout()
fig.savefig(run_dir / "mesh_preview.png", dpi=180)
plt.show()

## 8. Build the problem manually with `phast.Problem`

The fluent API is best while designing a model because each line corresponds to a physical decision. The region names on the left are the names used by this Python model; the `from_mesh` values are the physical names stored in `mesh.msh`.

The short run below uses AT2 damage, the spectral split, a quasi-static staggered solve, and CPU execution. The geometric notch is present in the mesh. Full damage pre-seeding is shown as an opt-in switch because it is useful for full fracture studies but can be too severe for a two-step Colab smoke test. Change only one choice at a time when exploring a new model.

In [ ]:
num_steps = 2          # quick tutorial run; increase for smoother damage evolution
u_top = 0.0002         # prescribed final displacement [mm]
l0 = 0.04             # phase-field length scale [mm]
enable_damage_preseed = False

problem = (
    phast.Problem("Notebook SENT tutorial")
    .mesh(str(mesh_path))
    .region("body", kind="domain", from_mesh="body")
    .region("bottom", from_mesh="bottom")
    .region("top", from_mesh="top")
    .region("notch", from_mesh="notch")
    .material(
        "glass",
        region="body",
        E=210000.0,
        nu=0.3,
        Gc=2.7,
        l0=l0,
        rho=7.8e-09,
        eta_residual=1.0e-07,
        energy_split="spectral",
        pf_model="AT2",
        plane_stress=False,
    )
    .boundary_condition("fix", region="bottom", dof="x", name="fix_bottom_x")
    .boundary_condition("fix", region="bottom", dof="y", name="fix_bottom_y")
    .boundary_condition("displacement", region="top", dof="y", value=1.0, name="pull_top")
    .analysis_step(
        "load",
        kind="quasi_static",
        controls={"protocol": "simple", "num_steps": num_steps, "dt": u_top / num_steps},
        active_boundary_conditions=["fix_bottom_x", "fix_bottom_y", "pull_top"],
    )
    .solver(
        "quasi_static",
        stagger_tol=1.0e-6,
        max_stagger=50,
        preconditioner="jacobi",
        damage_tol=1.0e-5,
        static_tol=1.0e-7,
        damage_max_iter=500,
        static_max_iter=500,
        fail_on_mechanics_nonconvergence=False,
        fail_on_stagger_nonconvergence=False,
        backend="auto",
        device="cpu",
    )
    .outputs(
        fields=[{"name": "trajectory", "every": 1, "format": "zarr"}],
        histories=[{"name": "reaction_force", "region": "bottom", "dof": "y"}],
        plots=True,
        profile=True,
        gif=True,
        gif_frames=24,
        gif_fields="damage",
        animation_format="gif",
        print_every=1,
    )
)

if enable_damage_preseed:
    problem.initial_condition("damage", region="notch", value=1.0)

problem.validate_setup()

## 9. Visualize supports, loading, and the damage seed

`problem.plot_setup(...)` writes a static setup preview. The extra overlay below explicitly marks the prescribed displacement and the seeded crack so the notebook remains readable even if the default preview style changes.

In [ ]:
setup_preview = run_dir / "initial_conditions.png"
problem.plot_setup(output=setup_preview)

img = np.asarray(PILImage.open(setup_preview).convert("RGB"))
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
ax.axis("off")
ax.set_title("PhAST setup preview")
plt.show()
print(setup_preview)

## 10. Solver settings: what the common choices mean

Phase-field fracture replaces a sharp crack with a continuous damage field `d`. Values near `0` represent intact material and values near `1` represent fully damaged material. The length scale `l0` controls the width of the diffuse crack band, so the mesh should resolve `l0` near the expected crack path.

| Choice | Typical values | Meaning |
|---|---|---|
| `pf_model` | `AT1`, `AT2` | Crack-density model. AT2 is smooth at damage onset; AT1 has a finite damage threshold and is common in dynamic benchmark decks. |
| `energy_split` | `spectral`, `amor`, `isotropic`, `star_convex` | How tensile and compressive elastic energy contributions are separated before driving damage. Use the split required by the benchmark or paper claim. |
| `solver_type` | `quasi_static`, `explicit` | Quasi-static solves coupled displacement/damage equilibrium at each load increment. Explicit dynamics advances the transient wave/crack problem with a stable time step. |
| static vs dynamic | `quasi_static` vs `explicit` | Static/quasi-static cases ignore inertia; dynamic cases require density and time-step controls. |
| implicit vs explicit | `quasi_static` uses iterative equilibrium solves; `explicit` uses time stepping | Current public examples promote quasi-static fracture and explicit dynamic fracture. Experimental or beta paths should stay clearly labelled. |
| trajectory format | `zarr`, `h5`, `both` | Zarr is preferred for public runs; HDF5 is retained for compatibility with older post-processing scripts. |

For a paper result, keep these choices in YAML and report them in the run manifest. Do not rely on notebook state as the only record of a simulation.

## 11. Write the runnable YAML configuration

The fluent `Problem` object is useful for authoring and setup validation. Its saved schema-v2 specification is an inspection representation and does not imply that every combination has a promoted execution adapter. This cell retains that specification separately, then writes the supported schema-v1 YAML fields used by the public fracture runner.


In [ ]:
problem.save(spec_path)

execution_config = {
    "schema_version": 1,
    "problem": {
        "name": "Notebook SENT workflow check",
        "reference": "Educational setup; not a validation benchmark",
    },
    "geometry": {"mesh_path": str(mesh_path)},
    "material": {
        "E": 210000.0,
        "nu": 0.3,
        "Gc": 2.7,
        "l0": l0,
        "rho": 7.8e-09,
        "eta_residual": 1.0e-07,
        "energy_split": "spectral",
        "pf_model": "AT2",
        "plane_stress": False,
    },
    "boundary_conditions": [
        {"nodes": "bottom", "type": "fix", "component": 0, "value": 0.0},
        {"nodes": "bottom", "type": "fix", "component": 1, "value": 0.0},
        {"nodes": "top", "type": "prescribe", "component": 1, "value": 1.0},
    ],
    "loading": {
        "protocol": "simple",
        "num_steps": num_steps,
        "dt": u_top / num_steps,
    },
    "solver": {
        "solver_type": "quasi_static",
        "stagger_tol": 1.0e-6,
        "max_stagger": 50,
        "preconditioner": "jacobi",
        "damage_tol": 1.0e-5,
        "static_tol": 1.0e-7,
        "damage_max_iter": 500,
        "static_max_iter": 500,
        "bounds_method": "post_clamp",
        "fail_on_mechanics_nonconvergence": False,
        "fail_on_stagger_nonconvergence": False,
        "backend": "auto",
    },
    "output": {
        "output_dir": str(output_dir),
        "trajectory": True,
        "trajectory_format": "zarr",
        "h5_every": 1,
        "plots": True,
        "profile": True,
        "gif": False,
        "print_every": 1,
        "reaction_node_set": "bottom",
        "reaction_component": 1,
    },
    "device": {"device": "cpu", "compile": False},
}
if enable_damage_preseed:
    execution_config["initial_conditions"] = {
        "preseed_notch_nodesets": ["notch"],
        "preseed_damage": 1.0,
    }

config_path.write_text(yaml.safe_dump(execution_config, sort_keys=False), encoding="utf-8")
print("Authoring specification:", spec_path)
print("Runnable configuration:", config_path)
print(config_path.read_text(encoding="utf-8")[:1600] + "\n...")

validate_cmd = [sys.executable, "-m", "phast", "run", str(config_path), "--validate-only"]
print(" ".join(validate_cmd))
subprocess.run(validate_cmd, check=True, cwd=repo_root)


## 12. Run the bounded solver workflow

The command below executes the runnable schema-v1 configuration and writes artifacts to `runs/notebook_sent/results/`. It is a two-step CPU workflow check, not evidence of crack initiation, propagation, mesh convergence, or benchmark agreement.


In [ ]:
run_cmd = [
    sys.executable, "-m", "phast", "run", str(config_path),
    "--output_dir", str(output_dir),
]
print(" ".join(run_cmd))
subprocess.run(run_cmd, check=True, cwd=repo_root)


## 13. Inspect the result directory

A completed public run should be inspectable without reading solver internals. The exact files depend on requested outputs, but the important classes are metadata/lockfiles, CSV histories, final PNGs, trajectory stores, and optional animations.

In [ ]:
for path in sorted(output_dir.iterdir()):
    if path.is_dir():
        print(f"{path.name}/")
    else:
        print(path.name)

result = phast.load_result(output_dir)
print("\nmetadata keys:", sorted(result.metadata().keys()))
print("histories:", result.history_names())
print("fields:", result.field_names())
print("visuals:", result.visuals())

## 14. Plot histories and final fields

The result API returns stored fields and histories. If a requested derived field was not stored directly, compute it during post-processing only when the required source fields are present.

In [ ]:
if "reaction_force" in result.history_names():
    rows = result.history("reaction_force")
elif "history" in result.history_names():
    rows = result.history("history")
else:
    rows = []

if rows:
    print(rows[:3])

def write_scalar_field_png(values, title, path, cmap="viridis"):
    arr = np.asarray(values)
    if arr.ndim > 1:
        arr = np.linalg.norm(arr, axis=-1)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    fig, ax = plt.subplots(figsize=(6, 5))
    if arr.size == points.shape[0]:
        im = ax.tripcolor(points[:, 0], points[:, 1], triangles, arr, shading="gouraud", cmap=cmap)
    elif arr.size == triangles.shape[0]:
        im = ax.tripcolor(points[:, 0], points[:, 1], triangles, facecolors=arr, edgecolors="none", cmap=cmap)
    else:
        plt.close(fig)
        print(f"Skipping {title}: field shape {values.shape} does not match nodes or elements")
        return None
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    fig.colorbar(im, ax=ax, shrink=0.82)
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)
    return path

for field_name, filename, title, cmap in [
    ("displacement", "displacement_final.png", "Final displacement magnitude", "magma"),
    ("stress", "stress_final.png", "Final stress magnitude", "plasma"),
    ("strain", "strain_final.png", "Final strain magnitude", "cividis"),
]:
    if result.has_field(field_name):
        write_scalar_field_png(result.field(field_name, step=-1), title, output_dir / filename, cmap=cmap)

candidate_images = [
    output_dir / "initial_conditions.png",
    run_dir / "initial_conditions.png",
    output_dir / "damage_final.png",
    output_dir / "displacement_final.png",
    output_dir / "stress_final.png",
    output_dir / "strain_final.png",
]
existing_images = [p for p in candidate_images if p.exists()]

if existing_images:
    fig, axes = plt.subplots(1, len(existing_images), figsize=(5 * len(existing_images), 4))
    axes = np.atleast_1d(axes)
    for ax, path in zip(axes, existing_images):
        ax.imshow(np.asarray(PILImage.open(path).convert("RGB")))
        ax.set_title(path.name)
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("No standard PNG fields were generated for this short run.")

## 15. Create or view animations

If the run generated `damage_evolution.gif` or `damage_evolution.mp4`, display it directly. For longer studies, prefer generating animations from the stored Zarr trajectory after the solve so rendering settings can be changed without rerunning the physics.

In [ ]:
from IPython.display import Image, Video, display

gif_path = output_dir / "damage_evolution.gif"
mp4_path = output_dir / "damage_evolution.mp4"

if not gif_path.exists() and result.has_field("damage"):
    history_len = len(result.history("results")) if "results" in result.history_names() else num_steps
    frames = []
    for step_index in range(history_len):
        damage = np.nan_to_num(result.field("damage", step=step_index), nan=0.0, posinf=1.0, neginf=0.0)
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.tripcolor(points[:, 0], points[:, 1], triangles, damage, shading="gouraud", cmap="inferno", vmin=0.0, vmax=1.0)
        ax.set_aspect("equal")
        ax.set_title(f"Damage, step {step_index}")
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("y [mm]")
        fig.colorbar(im, ax=ax, shrink=0.82)
        fig.tight_layout()
        fig.canvas.draw()
        frames.append(np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy())
        plt.close(fig)
    PILImage.fromarray(frames[0]).save(
        gif_path, save_all=True,
        append_images=[PILImage.fromarray(frame) for frame in frames[1:]],
        duration=800, loop=0,
    )

if gif_path.exists():
    display(Image(filename=str(gif_path)))
elif mp4_path.exists():
    display(Video(str(mp4_path), embed=True))
else:
    print("No damage animation could be generated because the damage trajectory is not available.")

## 16. What to change for a real study

1. Start from a public example closest to your target physics.
2. Keep the `.geo`, `.msh`, `config.yaml`, `run_fluent.py`, setup preview, final field images, animations, and CSV histories together.
3. Use a mesh size that resolves the phase-field length scale; a common starting point is several elements across `l0` near the crack path.
4. Validate the YAML with `python -m phast run config.yaml --validate-only` before using HPC time.
5. For dynamic fracture, switch the step and solver to `explicit`, include density, and verify the stable time step.
6. For quasi-static fracture, monitor staggered convergence and load-displacement response before trusting crack paths.
7. Archive the result directory, not only the notebook. The result directory is the reproducible simulation artifact.